In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

def crawl_compal_monthly_sales(start_year=2012, end_year=2025):
    """
    컴팔(Compal Electronics) 월간 매출 데이터 크롤링

    Parameters:
    start_year: 시작 연도
    end_year: 종료 연도

    Returns:
    DataFrame: 월간 매출 데이터
    """

    url = "https://www.compal.com/en-us/investor-relations/financial-release/"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        response.encoding = 'utf-8'

        soup = BeautifulSoup(response.text, 'html.parser')

        all_data = []

        for year in range(start_year, end_year + 1):
            # data-sale 속성으로 테이블 찾기
            table = soup.find('table', {'data-sale': str(year)})

            if not table:
                print(f"연도 {year} 데이터를 찾을 수 없습니다.")
                continue

            # thead에서 컬럼 정보 추출
            thead = table.find('thead')
            if not thead:
                continue

            header_cells = thead.find_all('td')
            if len(header_cells) < 5:
                continue

            current_year = header_cells[1].get_text(strip=True)
            prev_year = header_cells[2].get_text(strip=True)

            # tbody에서 데이터 추출
            tbody = table.find('tbody')
            if not tbody:
                continue

            rows = tbody.find_all('tr')

            for row in rows:
                cells = row.find_all('td')

                if not cells or len(cells) < 5:
                    continue

                month_text = cells[0].get_text(strip=True)

                # 분기(Q1, Q2, etc)와 연간(Annual) 데이터는 건너뛰기
                if month_text in ['1Q', '2Q', '3Q', '4Q', 'Annual']:
                    continue

                # 월 이름을 숫자로 변환
                month_mapping = {
                    'January': 1, 'February': 2, 'March': 3, 'April': 4,
                    'May': 5, 'June': 6, 'July': 7, 'August': 8,
                    'September': 9, 'October': 10, 'November': 11, 'December': 12
                }

                month_num = month_mapping.get(month_text)

                if not month_num:
                    continue

                # 데이터 추출
                try:
                    current_sales = cells[1].get_text(strip=True).replace(',', '')
                    prev_sales = cells[2].get_text(strip=True).replace(',', '')
                    mom = cells[3].get_text(strip=True).replace('%', '')
                    yoy = cells[4].get_text(strip=True).replace('%', '')

                    # 빈 값이 아닌 경우만 저장
                    if current_sales and current_sales != '':
                        data_dict = {
                            'year': year,
                            'month': month_num,
                            'date': f"{year}-{month_num:02d}",
                            'current_year': current_year,
                            'current_sales': float(current_sales) if current_sales else None,
                            'prev_year': prev_year,
                            'prev_sales': float(prev_sales) if prev_sales else None,
                            'mom': float(mom) if mom and mom != '--' else None,
                            'yoy': float(yoy) if yoy and yoy != '--' else None
                        }

                        all_data.append(data_dict)

                except (ValueError, IndexError) as e:
                    print(f"연도 {year}, 월 {month_text} 데이터 파싱 오류: {e}")
                    continue

            print(f"연도 {year} 데이터 수집 완료")
            time.sleep(0.5)

        if not all_data:
            print("크롤링된 데이터가 없습니다.")
            return pd.DataFrame()

        df = pd.DataFrame(all_data)

        # 날짜 순으로 정렬
        df = df.sort_values(['year', 'month']).reset_index(drop=True)

        print(f"\n총 {len(df)}개의 데이터를 수집했습니다.")
        print(f"기간: {df['date'].min()} ~ {df['date'].max()}")

        return df

    except requests.exceptions.RequestException as e:
        print(f"데이터 요청 중 오류 발생: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"예상치 못한 오류 발생: {e}")
        return pd.DataFrame()


def save_to_excel(df, filename='compal_monthly_sales.xlsx'):
    """
    데이터를 Excel 파일로 저장
    """
    if df.empty:
        print("저장할 데이터가 없습니다.")
        return

    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Monthly Sales', index=False)

            # 워크시트 가져오기
            worksheet = writer.sheets['Monthly Sales']

            # 컬럼 너비 자동 조정
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter

                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass

                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width

        print(f"\n데이터가 '{filename}' 파일로 저장되었습니다.")

    except Exception as e:
        print(f"Excel 저장 중 오류 발생: {e}")


def get_quarterly_summary(df):
    """
    분기별 요약 데이터 생성
    """
    if df.empty:
        return pd.DataFrame()

    # 분기 계산
    df['quarter'] = df['month'].apply(lambda x: f"Q{(x-1)//3 + 1}")

    # 분기별 집계
    quarterly = df.groupby(['year', 'quarter']).agg({
        'current_sales': 'sum',
        'prev_sales': 'sum'
    }).reset_index()

    # YoY 계산
    quarterly['yoy'] = ((quarterly['current_sales'] - quarterly['prev_sales']) /
                        quarterly['prev_sales'] * 100).round(2)

    return quarterly


# def main():
#     """
#     메인 실행 함수
#     """
#     print("컴팔(Compal Electronics) 월간 매출 데이터 크롤링 시작...")
#     print("=" * 60)
#
#     # 데이터 크롤링 (2012년부터 2025년까지)
#     df = crawl_compal_monthly_sales(start_year=2012, end_year=2025)
#
#     if not df.empty:
#         # 데이터 미리보기
#         print("\n" + "=" * 60)
#         print("데이터 미리보기 (최근 10개):")
#         print(df.tail(10).to_string())
#
#         # 기본 통계
#         print("\n" + "=" * 60)
#         print("기본 통계:")
#         print(df[['current_sales', 'mom', 'yoy']].describe())
#
#         # 분기별 요약
#         print("\n" + "=" * 60)
#         print("최근 분기별 요약:")
#         quarterly = get_quarterly_summary(df)
#         print(quarterly.tail(8).to_string())
#
#         # Excel 저장
#         save_to_excel(df, 'compal_monthly_sales.xlsx')
#
#         # 분기별 요약도 별도 시트로 저장
#         try:
#             with pd.ExcelWriter('compal_monthly_sales.xlsx',
#                               mode='a',
#                               engine='openpyxl',
#                               if_sheet_exists='replace') as writer:
#                 quarterly.to_excel(writer, sheet_name='Quarterly Summary', index=False)
#             print("분기별 요약 데이터도 저장되었습니다.")
#         except:
#             pass
#
#         return df
#     else:
#         print("크롤링 실패")
#         return None
#
#
# if __name__ == "__main__":
#     df = main()

In [2]:
df = crawl_compal_monthly_sales(start_year=2012, end_year=2025)
df

연도 2012 데이터 수집 완료
연도 2013 데이터 수집 완료
연도 2014 데이터 수집 완료
연도 2015 데이터 수집 완료
연도 2016 데이터 수집 완료
연도 2017 데이터 수집 완료
연도 2018 데이터 수집 완료
연도 2019 데이터 수집 완료
연도 2020 데이터 수집 완료
연도 2021 데이터 수집 완료
연도 2022 데이터 수집 완료
연도 2023 데이터 수집 완료
연도 2024 데이터 수집 완료
연도 2025 데이터 수집 완료

총 168개의 데이터를 수집했습니다.
기간: 2012-01 ~ 2025-12


,year,month,date,current_year,current_sales,prev_year,prev_sales,mom,yoy
0,2012,1,2012-01,2012,43872.0,2011,NaN,NaN,NaN
1,2012,2,2012-02,2012,58242.0,2011,NaN,32.8,NaN
2,2012,3,2012-03,2012,59640.0,2011,NaN,2.4,NaN
3,2012,4,2012-04,2012,50491.0,2011,NaN,-15.3,NaN
4,2012,5,2012-05,2012,51587.0,2011,NaN,2.2,NaN
...,...,...,...,...,...,...,...,...,...
163,2025,8,2025-08,2025,58809.0,2024,84111.0,0.4,-30.1
164,2025,9,2025-09,2025,69720.0,2024,83766.0,18.6,-16.8
165,2025,10,2025-10,2025,61920.0,2024,85452.0,-11.2,-27.5
166,2025,11,2025-11,2025,62968.0,2024,79666.0,1.7,-21.0
